In [1]:
# Day 4 - Data Cleaning: The most important real-world skill
# 80% of a data scientist's job is cleaning data
# Your SQL string functions all have Python equivalents

import pandas as pd
import numpy as np

# Create a realistic messy dataset — exactly what you'll get from production DBs
df = pd.DataFrame({
    "customer_id":  ["C001", "c002", "C003 ", " C004", "C005", "c006", "C007"],
    "name":         ["  Sam Kumar", "PRIYA SHARMA", "raj patel", "Anita Singh  ", 
                     "KIRAN RAO", "meera nair", "  ARJUN DAS  "],
    "phone":        ["9876543210", "98765-43211", "+91 9876543212", "987654321",  
                     "9876543214", "98765 43215", "9876543216"],
    "city":         ["Chennai", "mumbai", "DELHI", "Bangalore ", 
                     " hyderabad", "Chennai", "MUMBAI"],
    "salary":       ["45,000", "72000", "58,000.00", "91000", 
                     "63,000", "85000", "77,000"],
    "join_date":    ["2022-01-15", "15/03/2022", "March 10 2022", "2022-04-20",
                     "2022-05-01", "01-06-2022", "2022-07-11"],
    "email":        ["sam@gmail.com", "PRIYA@YAHOO.COM", "raj.patel@gmail.com",
                     "anita@", "kiran@company.com", "meera@gmail.com", "arjun@company.com"]
})

print("RAW MESSY DATA:")
print(df.to_string())
print(f"\nShape: {df.shape}")

RAW MESSY DATA:
  customer_id           name           phone        city     salary      join_date                email
0        C001      Sam Kumar      9876543210     Chennai     45,000     2022-01-15        sam@gmail.com
1        c002   PRIYA SHARMA     98765-43211      mumbai      72000     15/03/2022      PRIYA@YAHOO.COM
2       C003       raj patel  +91 9876543212       DELHI  58,000.00  March 10 2022  raj.patel@gmail.com
3        C004  Anita Singh         987654321  Bangalore       91000     2022-04-20               anita@
4        C005      KIRAN RAO      9876543214   hyderabad     63,000     2022-05-01    kiran@company.com
5        c006     meera nair     98765 43215     Chennai      85000     01-06-2022      meera@gmail.com
6        C007    ARJUN DAS        9876543216      MUMBAI     77,000     2022-07-11    arjun@company.com

Shape: (7, 7)


In [2]:
# Day 4 - String cleaning
# SQL: TRIM, UPPER, LOWER, REPLACE — all here

# 1. customer_id — strip spaces and uppercase
df["customer_id"] = df["customer_id"].str.strip().str.upper()

# 2. name — strip spaces and title case
df["name"] = df["name"].str.strip().str.title()

# 3. city — strip spaces and title case
df["city"] = df["city"].str.strip().str.title()

# 4. salary — remove commas and .00, convert to int
df["salary"] = df["salary"].str.replace(",", "").str.replace(".00", "").astype(int)

# 5. email — lowercase, flag invalid emails
df["email"] = df["email"].str.lower()
df["email_valid"] = df["email"].str.contains(r"^[\w\.-]+@[\w\.-]+\.\w+$", regex=True)

# 6. phone — extract only digits, flag invalid (not 10 digits)
df["phone_clean"] = df["phone"].str.replace(r"\D", "", regex=True)
df["phone_clean"] = df["phone_clean"].str.replace(r"^91", "", regex=True)
df["phone_valid"] = df["phone_clean"].str.len() == 10

print("CLEANED DATA:")
print(df[["customer_id", "name", "city", "salary", 
          "email_valid", "phone_valid"]].to_string())
print("\nInvalid emails:")
print(df[~df["email_valid"]][["name", "email"]])
print("\nInvalid phones:")
print(df[~df["phone_valid"]][["name", "phone", "phone_clean"]])

CLEANED DATA:
  customer_id          name       city  salary  email_valid  phone_valid
0        C001     Sam Kumar    Chennai   45000         True         True
1        C002  Priya Sharma     Mumbai   72000         True         True
2        C003     Raj Patel      Delhi   58000         True         True
3        C004   Anita Singh  Bangalore   91000        False        False
4        C005     Kiran Rao  Hyderabad   63000         True         True
5        C006    Meera Nair    Chennai   85000         True         True
6        C007     Arjun Das     Mumbai   77000         True         True

Invalid emails:
          name   email
3  Anita Singh  anita@

Invalid phones:
          name      phone phone_clean
3  Anita Singh  987654321   987654321


In [6]:
# Day 4 - Datetime handling (fixed for newer Pandas)

print("Raw join dates:")
print(df["join_date"].values)

# format='mixed' handles different formats in the same column
df["join_date_clean"] = pd.to_datetime(df["join_date"], format="mixed", dayfirst=True)

print("\nCleaned join dates:")
print(df["join_date_clean"])

# Extract components — like EXTRACT() in SQL
df["join_year"]       = df["join_date_clean"].dt.year
df["join_month"]      = df["join_date_clean"].dt.month
df["join_month_name"] = df["join_date_clean"].dt.strftime("%B")
df["join_day"]        = df["join_date_clean"].dt.day
df["join_weekday"]    = df["join_date_clean"].dt.day_name()

# Days since joining — like DATEDIFF in SQL
df["days_employed"] = (pd.Timestamp.today() - df["join_date_clean"]).dt.days

print("\nDate features:")
print(df[["name", "join_date", "join_date_clean",
          "join_month_name", "join_weekday", "days_employed"]].to_string())

Raw join dates:
['2022-01-15' '15/03/2022' 'March 10 2022' '2022-04-20' '2022-05-01'
 '01-06-2022' '2022-07-11']

Cleaned join dates:
0   2022-01-15
1   2022-03-15
2   2022-03-10
3   2022-04-20
4   2022-05-01
5   2022-06-01
6   2022-07-11
Name: join_date_clean, dtype: datetime64[ns]

Date features:
           name      join_date join_date_clean join_month_name join_weekday  days_employed
0     Sam Kumar     2022-01-15      2022-01-15         January     Saturday           1561
1  Priya Sharma     15/03/2022      2022-03-15           March      Tuesday           1502
2     Raj Patel  March 10 2022      2022-03-10           March     Thursday           1507
3   Anita Singh     2022-04-20      2022-04-20           April    Wednesday           1466
4     Kiran Rao     2022-05-01      2022-05-01             May       Sunday           1455
5    Meera Nair     01-06-2022      2022-06-01            June    Wednesday           1424
6     Arjun Das     2022-07-11      2022-07-11            July 

In [8]:
# Day 4 - Regex on DataFrame (fixed)

logs = pd.DataFrame({
    "log": [
        "User 9876543210 placed order ORD-2024-001 worth ₹1,500",
        "User 8765432109 cancelled order ORD-2024-002 worth ₹750",
        "User 7654321098 placed order ORD-2024-003 worth ₹2,250",
    ]
})

# .str.extract() returns a DataFrame — squeeze to Series with [0]
logs["phone"]    = logs["log"].str.extract(r"(\b[6-9]\d{9}\b)")[0]
logs["order_id"] = logs["log"].str.extract(r"(ORD-\d{4}-\d{3})")[0]
logs["amount"]   = logs["log"].str.extract(r"₹([\d,]+)")[0].str.replace(",","").astype(int)

print(logs[["phone", "order_id", "amount"]])

        phone      order_id  amount
0  9876543210  ORD-2024-001    1500
1  8765432109  ORD-2024-002     750
2  7654321098  ORD-2024-003    2250


In [9]:
# Day 4 - Final deliverable: Production data quality pipeline

import pandas as pd
import numpy as np
import re

def clean_dataframe(df):
    """
    Production data cleaning pipeline.
    Handles: whitespace, casing, nulls, duplicates, 
             invalid emails, invalid phones, salary formatting
    """
    print("="*50)
    print("DATA CLEANING PIPELINE")
    print("="*50)
    print(f"Input: {df.shape[0]} rows × {df.shape[1]} columns")
    
    original_rows = len(df)
    report = {}

    # 1. Strip whitespace from all string columns
    str_cols = df.select_dtypes(include="object").columns
    for col in str_cols:
        df[col] = df[col].str.strip()
    report["whitespace_cleaned"] = len(str_cols)

    # 2. Standardize casing
    if "name" in df.columns:
        df["name"] = df["name"].str.title()
    if "city" in df.columns:
        df["city"] = df["city"].str.title()
    if "email" in df.columns:
        df["email"] = df["email"].str.lower()
    if "customer_id" in df.columns:
        df["customer_id"] = df["customer_id"].str.upper()

    # 3. Clean salary
    if "salary" in df.columns:
        df["salary"] = (df["salary"].astype(str)
                        .str.replace(",", "")
                        .str.replace(".00", "")
                        .str.strip())
        df["salary"] = pd.to_numeric(df["salary"], errors="coerce")
        report["salary_nulls_after_clean"] = df["salary"].isna().sum()

    # 4. Validate email
    if "email" in df.columns:
        email_pattern = r"^[\w\.-]+@[\w\.-]+\.\w+$"
        df["email_valid"] = df["email"].str.match(email_pattern)
        report["invalid_emails"] = (~df["email_valid"]).sum()

    # 5. Validate phone
    if "phone" in df.columns:
        df["phone_clean"] = (df["phone"].astype(str)
                             .str.replace(r"\D", "", regex=True)
                             .str.replace(r"^91", "", regex=True))
        df["phone_valid"] = df["phone_clean"].str.len() == 10
        report["invalid_phones"] = (~df["phone_valid"]).sum()

    # 6. Remove duplicates
    before_dedup = len(df)
    df = df.drop_duplicates()
    report["duplicates_removed"] = before_dedup - len(df)

    # 7. Null report
    null_counts = df.isna().sum()
    report["total_nulls"] = null_counts.sum()

    # Print report
    print("\n📋 CLEANING REPORT:")
    for key, val in report.items():
        print(f"  {key}: {val}")
    
    print(f"\n✅ Output: {len(df)} rows × {df.shape[1]} columns")
    print(f"   Rows removed: {original_rows - len(df)}")
    print("="*50)
    
    return df


# Test it on our messy dataset
raw_df = pd.DataFrame({
    "customer_id": ["C001", "c002", "C003 ", " C004", "C005", "c006", "C007", "C001"],
    "name":        ["  Sam Kumar", "PRIYA SHARMA", "raj patel", "Anita Singh  ",
                    "KIRAN RAO", "meera nair", "  ARJUN DAS  ", "  Sam Kumar"],
    "phone":       ["9876543210", "98765-43211", "+91 9876543212", "987654321",
                    "9876543214", "98765 43215", "9876543216", "9876543210"],
    "city":        ["Chennai", "mumbai", "DELHI", "Bangalore ",
                    " hyderabad", "Chennai", "MUMBAI", "Chennai"],
    "salary":      ["45,000", "72000", "58,000.00", "91000",
                    "63,000", "85000", "77,000", "45,000"],
    "email":       ["sam@gmail.com", "PRIYA@YAHOO.COM", "raj.patel@gmail.com",
                    "anita@", "kiran@company.com", "meera@gmail.com",
                    "arjun@company.com", "sam@gmail.com"]
})

cleaned_df = clean_dataframe(raw_df)
print("\nCLEANED DATA:")
print(cleaned_df[["customer_id", "name", "city", 
                   "salary", "email_valid", "phone_valid"]].to_string())

# Save cleaned data
cleaned_df.to_csv("cleaned_customers.csv", index=False)
print("\nSaved: cleaned_customers.csv")

DATA CLEANING PIPELINE
Input: 8 rows × 6 columns

📋 CLEANING REPORT:
  whitespace_cleaned: 6
  salary_nulls_after_clean: 0
  invalid_emails: 1
  invalid_phones: 1
  duplicates_removed: 1
  total_nulls: 0

✅ Output: 7 rows × 9 columns
   Rows removed: 1

CLEANED DATA:
  customer_id          name       city  salary  email_valid  phone_valid
0        C001     Sam Kumar    Chennai   45000         True         True
1        C002  Priya Sharma     Mumbai   72000         True         True
2        C003     Raj Patel      Delhi   58000         True         True
3        C004   Anita Singh  Bangalore   91000        False        False
4        C005     Kiran Rao  Hyderabad   63000         True         True
5        C006    Meera Nair    Chennai   85000         True         True
6        C007     Arjun Das     Mumbai   77000         True         True

Saved: cleaned_customers.csv
